# The de Rham complex

Different types of finite elements, connected by differential operators:

$$
H^1 \; \xrightarrow{\;\nabla\;} \; H(\operatorname{curl}) \; \xrightarrow{\;\operatorname{curl}\;} \; H(\operatorname{div}) \; \xrightarrow{\;\operatorname{div}\;} \; L^2
$$

| nodal elements | edge elements | face elements | cell elements |
|:-:|:-:|:-:|:-:|
| <img src="../pictures/nodalelement.png" width="150"> | <img src="../pictures/edgeelement.png" width="150"> | <img src="../pictures/faceelement.png" width="150"> | <img src="../pictures/l2element.png" width="150"> |
| φ | E, H, A | j, B, D | ρ |

Continuous and discrete spaces satisfy the **exact sequence property**

$$
\operatorname{range}(\nabla) = \ker(\operatorname{curl}), \qquad \operatorname{range}(\operatorname{curl}) = \ker(\operatorname{div}).
$$

In particular, $B = \operatorname{curl} A$ with $A$ from the edge element space is exactly divergence-free, as in the [coil](coil.ipynb) example.

## Exact discretization of conservation laws

Charge conservation $\frac{\partial \rho}{\partial t} + \operatorname{div} j = 0$ holds exactly on every cell, since normal components of $j$ are single-valued on faces:

$$
\int\limits_{\text{cell } T} \frac{\partial \rho}{\partial t} = - \int\limits_T \operatorname{div} j = -\sum_{\text{faces } F \subset \partial T} \; \int\limits_F j \cdot n
$$

Faraday's law $\frac{\partial B}{\partial t} = -\operatorname{curl} E$ holds exactly on every face, since tangential components of $E$ are single-valued on edges:

$$
\int\limits_{\text{face } F} \frac{\partial B}{\partial t} \cdot n = - \int\limits_F \operatorname{curl} E \cdot n = -\sum_{\text{edges } e \subset \partial F} \; \int\limits_e E \cdot \tau
$$

# The Piola transformation

The space $H(\operatorname{div}) = \{ \sigma \in [L_2(\Omega)]^d : \operatorname{div} \sigma \in L_2 \}$ has a continuous normal trace:
functions have single-valued normal components across element interfaces.
Discrete subspaces are the Brezzi-Douglas-Marini ${\mathcal BDM}_k$ or Raviart-Thomas elements.

For a mapping of domains $x = \Phi(\hat x)$ with Jacobian $F = \Phi^\prime$ and determinant $J = \det F$, vector fields are pushed forward by the Piola transformation

$$
{\mathcal P} \hat \sigma (x) := \frac{1}{J} F \hat \sigma(\hat x).
$$

It preserves normal fluxes, and divergence-free fields stay **exactly** divergence-free:
$\operatorname{div} {\mathcal P} \hat \sigma = \frac{1}{J} \widehat{\operatorname{div}} \, \hat \sigma$.

A parabolic flow profile in a channel, represented in $H(\operatorname{div})$:

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
import ipywidgets as widgets

mesh = MoveTo(0,-0.1).Rectangle(1,0.2).Face().GenerateMesh(dim=2, maxh=0.05)

fes = HDiv(mesh, order=2)
sigma = GridFunction(fes)
sigma.Set ( (2-100*y*y, 0))

Bend the channel and change its thickness. The field is transported with the mesh by the Piola transformation:

In [ ]:
disp = GridFunction(VectorH1(mesh, order=2))
scene = Draw (sigma, vectors={"grid_size":30}) 

def Deformation(bend, thick):
    return (-2*bend*x*y-0.5*y*y*(thick-1), bend*x*x + x*y*(thick-1))

def myfunc(bend, thick): 
    disp.Set(Deformation(bend, thick))
    mesh.SetDeformation(disp)
    scene.Redraw()
    mesh.UnsetDeformation()

params = widgets.interactive(myfunc,
        bend=widgets.FloatSlider(value=0, min=-0.2, max=0.2, step=0.001, description='bend:',
                                 continuous_update=True, readout_format='.3f'),
        thick=widgets.FloatSlider(value=1, min=0.01, max=2, step=0.001, description='thick:',
                                  continuous_update=True, readout_format='.3f'))
display (params)

For a fixed deformation we compare with a vector field transported by values (`VectorH1`):
only the Piola-mapped field stays divergence-free.

In [ ]:
disp.Set(Deformation(bend=0.15, thick=0.5))

for space in [HDiv(mesh, order=2), VectorH1(mesh, order=2)]:
    gf = GridFunction(space)
    gf.Set ( (2-100*y*y, 0))
    mesh.SetDeformation(disp)
    print (f"{type(space).__name__:10s} |div| on deformed domain = {sqrt(Integrate(div(gf)**2, mesh)):.2e}")
    mesh.UnsetDeformation()

mesh.SetDeformation(disp)
Draw (sigma, vectors={"grid_size":30})
mesh.UnsetDeformation()

Applications of $H(\operatorname{div})$-conforming velocities:

* exactly divergence-free flow fields, as in the [Tesla valve](tesla.ipynb)
* domain morphing and fluid-structure interaction: the divergence constraint survives the mesh motion
* model order reduction on parametrized geometries